# UAV External Subsystems

In order to expand Aviary's modeling capabilities to include small aircraft, the UAV-specific mass, propulsion, and aerodynamics external subsytems were created. This document will go over the important points of these subsystems, starting with their location in the Aviary repo.

All three subsystems' models and tests are located in the aviary/models/external_subsystems folder inside the Aviary repository. The UAV .csv file as well as example files can be found in the aviary/models/aircraft/small_scale_uav folder. Lastly, the UAV phase info file can be located in the aviary/models/missions folder.

## The Mass Subsystem

The mass subsystem calculates and sums the mass of four components of the aircraft: wing, horizontal tail, vertical tail, and fuselage. A separate variable hierarchy was defined for the structural aspects of these components; they are listed below:

In [ ]:

#Variables specific to FUSELAGE mass:
AVG_HEIGHT = 'aircraft:fuselage:average_height'
AVG_WIDTH = 'aircraft:fuselage:average_width'
BULKHEAD_DENSITY = 'aircraft:fuselage:bulkhead_density'
BULKHEAD_LIGHTENING_FACTOR = 'aircraft:fuselage:bulkhead_lightening_factor'
BULKHEAD_MATERIALS = 'aircraft:fuselage:bulkhead_materials'
BULKHEAD_THICKNESS = 'aircraft:fuselage:bulkhead_thickness'
NUM_BULKHEADS = 'aircraft:fuselage:number_of_bulkheads'
FLOOR_DENSITY = 'aircraft:fuselage:floor_density'
FLOOR_LENGTH = 'aircraft:fuselage:floor_length'
FLOOR_THICKNESS = 'aircraft:fuselage:floor_thickness'

#Variables specific to simple WING mass:
FOAM_DENSITY = 'aircraft:wing:foam_density'
ROD_DENSITY = 'aircraft:wing:rod_density'
ROD_RADIUS = 'aircraft:wing:rod_radius'
ROD_THICKNESS = 'aircraft:wing:rod_thickness'
TYPE = 'aircraft:wing:type'

#Variables that are used for HORIZONTAL TAIL, VERTICAL TAIL, and WING mass, but NOT used for FUSELAGE:
AIRFOIL_PATH = 'aircraft:horizontal_tail:airfoil_path'
NUM_RIBS = 'aircraft:horizontal_tail:number_of_ribs'
NUM_STRINGERS = 'aircraft:horizontal_tail:number_of_stringers'
RIB_DENSITY = 'aircraft:horizontal_tail:rib_density'
RIB_LIGHTENING_FACTOR = 'aircraft:horizontal_tail:rib_lightening_factor'
RIB_MATERIALS = 'aircraft:horizontal_tail:rib_materials'
RIB_THICKNESS = 'aircraft:horizontal_tail:rib_thickness'

#Variables that every mass calculation uses. 'fuselage' in the strings 
# below can be fuselage/horizontal_tail/vertical_tail/wing, these variables are
#shared between all four masses
GLUE_FACTOR = 'aircraft:fuselage:glue_factor'
MISC_MASS = 'aircraft:fuselage:misc_mass'
NUM_SPARS = 'aircraft:fuselage:number_of_spars'
SHEETING_COVERAGE = 'aircraft:fuselage:sheeting_coverage'
SHEETING_DENSITY = 'aircraft:fuselage:sheeting_density'
SHEETING_LIGHTENING_FACTOR = 'aircraft:fuselage:sheeting_lightening_factor'
SHEETING_THICKNESS = 'aircraft:fuselage:sheeting_thickness'
AREAL_SKIN_DENSITY = 'aircraft:fuselage:areal_skin_density'
SPAR_DENSITY = 'aircraft:fuselage:spar_density'
SPAR_OUTER_DIAMETER = 'aircraft:fuselage:spar_outer_diameter'
SPAR_WALL_THICKNESS = 'aircraft:fuselage:spar_wall_thickness'
STRINGER_DENSITY = 'aircraft:fuselage:stringer_density'
STRINGER_THICKNESS = 'aircraft:fuselage:stringer_thickness'

### The WingType Option
The UAV wing mass calculation provides the WingType option (of variable type enum), that specifies whether the wing mass calculation is for a traditional UAV wing ('medium') or a solid foam wing with two spars/rods running through it ('simple'). The simple wing option is particularly useful for Design-Build-Fly teams that follow this design. The snippets below show the enum definition as well as the variables that are needed for simple wing.

In [ ]:
from enum import Enum

class WingType(Enum):
    '''
    Specifies the type of wing used in the UAV mass wing model computation.
    
    SIMPLE:
        Wing is made from solid foam and two hollow spars
    MEDIUM: 
        Wing design includes spars, sheeting, stringers, ribs, ..., and is hollow
    '''

    SIMPLE = 'simple'
    MEDIUM = 'medium'

In [ ]:
'''These are the variables needed for a simple wing mass calculation'''

FOAM_DENSITY = 'aircraft:wing:foam_density'
ROD_DENSITY = 'aircraft:wing:rod_density'
ROD_RADIUS = 'aircraft:wing:rod_radius'
ROD_THICKNESS = 'aircraft:wing:rod_thickness'

SPAN = 'aircraft:wing:span'
ROOT_CHORD = 'aircraft:wing:root_chord'
AIRFOIL_PATH = 'airfract:wing:airfoil_path'


## The Propulsion Subsystem
The propulsion model consists of three files: premission, mission, and performance. The performance file contains all of the ExplicitComponents that will be used in the subsystem. The premission file creates a group of motor and battery calculations and the mission file calculates the ODE of the motor. The mission file also adds constraints on RPM and energy that prevent unrealistic outputs.

## The Aerodynamics Subsystem
The UAV aerodynamics model consists of two main files. The aero_OAS_analysis.py uses an atmospheric ExplicitComponent to output necessary quantities for OpenAeroStruct and then uses OpenAeroStruct to collect lift and drag values. The aero_model.py file uses ExplicitComponents to calculate the lift and drag of the aircraft's non lifting surfaces (vertical tail, fuselage) and then groups all relevant subsystems from both files together. 

### The AlphaComp Component
The AlphaComp ExplicitComponent is defined in the aero_OAS_analysis file and is used to achieve a desirable angle of attack via the slack variable: 'lift_balance_residual'. The component take in overall lift and mass and subtracts mass*gravity from lift to get the residual. Later in the model this residual is constrained to zero, restricting angle of attack to the value that leads to flight at a constant altitude. See below:

In [ ]:
import openmdao.api as om
import numpy as np
from aviary.variable_info.functions import add_aviary_input, add_aviary_option
from aviary.models.external_subsystems.UAV.UAV_variable_info.UAV_variables import Dynamic, Mission

class AlphaComp(om.ExplicitComponent):

    def initialize(self):
        self.options.declare('num_nodes', types=int)
        add_aviary_option(self, Mission.GRAVITY, units='m/s**2')

    def setup(self):
        nn = self.options['num_nodes']
        rows_cols = np.arange(nn)
        add_aviary_input(self, Dynamic.Vehicle.LIFT, shape=nn, units='N')
        add_aviary_input(self, Dynamic.Vehicle.MASS, shape=nn, units='kg')
        self.add_input( 'alpha', val=np.full(nn, 3.0), units='deg', )

        # This output will be constrained to zero.
        self.add_output( 'lift_balance_residual', val=np.zeros(nn),  units='N', desc='Lift equilibrium residual', )
        self.declare_partials('lift_balance_residual', Dynamic.Vehicle.LIFT, rows=rows_cols, cols=rows_cols)
        self.declare_partials('lift_balance_residual', Dynamic.Vehicle.MASS, rows=rows_cols, cols=rows_cols)
        self.declare_partials('lift_balance_residual', 'alpha', rows=rows_cols, cols=rows_cols)

    def compute(self, inputs, outputs):
        L = inputs[Dynamic.Vehicle.LIFT]
        m = inputs[Dynamic.Vehicle.MASS]
        g = self.options[Mission.GRAVITY][0] # m/s**2
        outputs['lift_balance_residual'] = L - (m * g)

One thing the aerodynamics subsystem is lacking is ample tests. Right now only one test exists, and it is for the builder. It would be very useful in the future to have tests for each of the lower level components in this subsystem.

## Mars and Min Energy Examples
There are two example files in the small_scale_UAV folder that run to varying degrees of success. The min energy example uses the external mass, propulsion, and aerodynamics subsystems to optimize for the lowest energy cruise over a fixed distance for a UAV aircraft.

The mars example attempts to minimize the time it takes a UAV aircraft to complete a cruise at a fixed distance and altitude near the bottom of the Hellas Basin on Mars. There are several standing issues with this optimization, chief among them that it only runs if wing span is removed as a design variable entirely. This is not desirable, as wing span should be a design variable and is used in other optimizations successfully.

Further issues are evident once the optimization is ran without the presence of wing span: 
 - The optimization runs, but exceeds max iterations and fails
 - Lift is far lower than it should be, around 3 N
 - NaN presence - likely from dividing by zero in places
 - Structure mass and cruise mass output different values but should be equal


